# BT4012: Student-selected pairwise XGBoost ranking model

**11 October 2026.** After reviewing the earlier ensemble result, the student selected a new pairwise-ranking candidate and retained the four chronological checks.

This is a newly trained model with a different learning objective. It is not a reuse of the previous classifier's fitted trees or an average of the earlier models. The best confirmed public score remains 0.94265 until the student submits a candidate and records its actual Kaggle score.

Only the supplied competition CSVs are used. Unknown labels are excluded from supervised training. No external labels or competition solutions are used. The settings below are fixed; no automatic parameter search or automatic final-model selection is performed.

## 1. What is new, and why test it?

The earlier classifier learns **how accurately it can predict probabilities** using binary logistic loss. Setting its evaluation metric to AUC monitors ranking performance; it does not change the underlying training loss into an AUC loss.

The new ranking model learns from pairs: **a known illicit transaction should receive a higher score than a known licit transaction**. When that order is wrong, the pair contributes a training penalty. Pairs with the same label do not tell the model which transaction should rank higher.

AUC measures the fraction of illicit–licit pairs that are ordered correctly, with half credit for ties. Pairwise logistic loss is a smooth training surrogate for ordering; it does not optimize the exact, discrete AUC directly and cannot guarantee better AUC.

The model uses `rank:pairwise` with random pair sampling. Eight candidate comparisons per row provide more opportunities to learn orderings than one, while keeping CPU computation bounded. Same-label comparisons are skipped.

All rows inside each training partition form one ranking group. This allows comparisons across time steps because Kaggle's pooled AUC compares transactions across its entire test set. `time_step`, group identifiers, and transaction identifiers are not prediction features.

The ranker returns an ordering score, not a calibrated illicit probability. A strictly increasing sigmoid converts its scores into the interval 0–1 for the experimental CSV. We check that this conversion preserves validation AUC. Those bounded scores should not be interpreted as calibrated probabilities.

[XGBoost ranking documentation](https://xgboost.readthedocs.io/en/release_3.2.0/tutorials/learning_to_rank.html), [AUC and ranking parameters](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html).

## 2. Student-selected settings and comparison limits

Both models use all 165 original feature columns, 15 leaves, 127 bins, learning rate 0.05, at most 400 trees, a fixed seed of 42, and two CPU threads. There is no time-feature removal or feature sampling in this experiment.

The original control retains its previous regularization (lambda 10, minimum child weight 3). The ranker uses standard ranking defaults (lambda 1, minimum child weight 1), including the default ranking normalization settings. These were part of the proposed fixed candidate the student selected. Different losses produce different gradient scales, so the same numeric penalty does not necessarily have the same effect.

This evaluates the complete ranking candidate against the selected classifier. Any difference cannot be attributed exclusively to the loss change, because the regularization values also differ. No values will be changed after seeing the results.

The four checks train through 14, 17, 19, and 21 and predict the next fourteen steps. Within each training period, its last three steps choose the number of trees. The outer checking answers do not choose that number. The outer checks overlap and reuse previously examined periods, so they are development evidence rather than independent final tests.

**Implementation correction:** the initial implementation used XGBoost's built-in ranking `auc`. On these binary labels it did not match the desired binary ROC-AUC: for the first inner period, the first tree scored 0.669652 under the built-in ranking metric and 0.972786 under binary ROC-AUC. That mismatch led to unsuitable stopping rounds. This corrected implementation uses the same binary ROC-AUC calculation as outer evaluation, explicitly maximizes it, and verifies the reported stopping score against a separate calculation. The model settings and chronological checks remain those the student selected. The initial implementation is retained separately for the development record.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import xgboost
from scipy.special import expit
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier, XGBRanker
from xgboost.callback import EarlyStopping

locations = [Path.cwd() / 'Data Set', Path.cwd().parent / 'Data Set',
             Path('/Users/mlvlo/BT4012/Kaggle/Data Set')]
DATA_DIR = next((p for p in locations if (p / 'Train Data.csv').is_file()), None)
assert DATA_DIR is not None, 'Set DATA_DIR to the provided competition CSVs.'
OUTPUT_DIR = Path.cwd() / 'pairwise_ranking_results'
OUTPUT_DIR.mkdir(exist_ok=True)
FEATURES = [f'feat_{i}' for i in range(1, 166)]
CHECKS = [(14, 15, 28), (17, 18, 31), (19, 20, 33), (21, 22, 35)]
MAXIMUM_ROUNDS, STOPPING_PATIENCE, CPU_THREADS = 400, 60, 2
PAIR_COMPARISONS, RANDOM_SEED = 8, 42

train = pd.read_csv(DATA_DIR / 'Train Data.csv',
                    usecols=['txId', 'time_step', 'label'] + FEATURES)
assert train.txId.is_unique and train.time_step.between(1, 35).all()
known = train.loc[train.label.notna()].copy()
assert known.label.isin([0, 1]).all()
known['label'] = known.label.astype(int)
assert known.label.nunique() == 2
assert np.isfinite(known[FEATURES].to_numpy()).all()
print('Known training labels:', len(known), '; illicit:', int(known.label.sum()))
print('XGBoost:', xgboost.__version__, '; CPU threads:', CPU_THREADS)


Known training labels: 31235 ; illicit: 3644
XGBoost: 3.2.0 ; CPU threads: 2


## 3. Train fresh models with chronological stopping

In the first practice exam, initial fitting uses steps 1–11 and stopping uses 12–14. After selecting the number of trees, the model is refitted from scratch on 1–14 and predicts 15–28. The other cutoffs follow the same three-step stopping rule.

The additional ranker's group argument describes which training rows may form pairs. It never includes the outer checking rows or their answers.

In [2]:
def binary_roc_auc(labels, scores):
    return float(roc_auc_score(labels, scores))


def make_model(family, rounds, *, stopping=False):
    common = dict(
        n_estimators=int(rounds), learning_rate=0.05, tree_method='hist', device='cpu',
        grow_policy='lossguide', max_depth=0, max_leaves=15, max_bin=127,
        subsample=1.0, colsample_bytree=1.0,
        eval_metric='auc', early_stopping_rounds=STOPPING_PATIENCE if stopping else None,
        n_jobs=CPU_THREADS, random_state=RANDOM_SEED,
    )
    if family == 'original_xgboost':
        return XGBClassifier(**common, objective='binary:logistic',
                             reg_lambda=10, min_child_weight=3, scale_pos_weight=1.0)
    assert family == 'pairwise_ranker'
    common['eval_metric'] = binary_roc_auc
    common['early_stopping_rounds'] = None
    common['callbacks'] = [EarlyStopping(
        rounds=STOPPING_PATIENCE, metric_name='binary_roc_auc',
        data_name='validation_0', maximize=True,
    )] if stopping else None
    return XGBRanker(**common, objective='rank:pairwise',
                     reg_lambda=1, min_child_weight=1,
                     lambdarank_pair_method='mean',
                     lambdarank_num_pair_per_sample=PAIR_COMPARISONS,
                     lambdarank_normalization=True,
                     lambdarank_score_normalization=True)


def fit_model(family, learning):
    end = int(learning.time_step.max())
    fitting = learning.loc[learning.time_step <= end - 3]
    stopping = learning.loc[learning.time_step > end - 3]
    assert fitting.time_step.max() < stopping.time_step.min()
    assert fitting.label.nunique() == stopping.label.nunique() == 2
    with threadpool_limits(limits=CPU_THREADS):
        initial = make_model(family, MAXIMUM_ROUNDS, stopping=True)
        options = dict(eval_set=[(stopping[FEATURES], stopping.label)], verbose=False)
        if family == 'pairwise_ranker':
            options.update(group=[len(fitting)], eval_group=[[len(stopping)]])
        initial.fit(fitting[FEATURES], fitting.label, **options)
        rounds = int(initial.best_iteration + 1)
        assert 1 <= rounds <= MAXIMUM_ROUNDS
        inner_scores = (initial.predict(stopping[FEATURES]) if family == 'pairwise_ranker'
                        else initial.predict_proba(stopping[FEATURES])[:, 1])
        independently_calculated_auc = float(roc_auc_score(stopping.label, inner_scores))
        # XGBoost serializes Python custom metrics to six decimal places.
        metric_tolerance = 5.01e-7 if family == 'pairwise_ranker' else 1e-12
        assert abs(independently_calculated_auc - float(initial.best_score)) < metric_tolerance
        metric_name = 'binary_roc_auc' if family == 'pairwise_ranker' else 'auc'
        curve = initial.evals_result()['validation_0'][metric_name]
        pd.DataFrame({'round': np.arange(1, len(curve) + 1), 'inner_binary_roc_auc': curve}).to_csv(
            OUTPUT_DIR / f'stopping_curve_{family}_{end}.csv', index=False)
        refitted = make_model(family, rounds)
        options = {'group': [len(learning)]} if family == 'pairwise_ranker' else {}
        refitted.fit(learning[FEATURES], learning.label, **options)
    return refitted, rounds


def predict_scores(family, model, rows):
    with threadpool_limits(limits=CPU_THREADS):
        if family == 'original_xgboost':
            return model.predict_proba(rows[FEATURES])[:, 1]
        return model.predict(rows[FEATURES])


## 4. Train and evaluate the new ranking model

Use exactly the same checking transactions for the ranker and original classifier. AUC can compare their ordering even though one output is a probability and the other is an unbounded score. Save both the raw ranker score and its bounded sigmoid conversion.

In [3]:
records, saved = [], {}
started_all = time.perf_counter()
for end, first, last in CHECKS:
    learning = known.loc[known.time_step <= end]
    checking = known.loc[known.time_step.between(first, last)]
    assert learning.time_step.max() < checking.time_step.min()
    assert set(learning.txId).isdisjoint(checking.txId)
    assert checking.label.nunique() == 2
    saved[f'labels_{end}'] = checking.label.to_numpy()
    saved[f'txIds_{end}'] = checking.txId.to_numpy()
    saved[f'time_steps_{end}'] = checking.time_step.to_numpy()
    for family in ['original_xgboost', 'pairwise_ranker']:
        started = time.perf_counter()
        fitted, rounds = fit_model(family, learning)
        scores = predict_scores(family, fitted, checking)
        assert np.isfinite(scores).all()
        auc = float(roc_auc_score(checking.label, scores))
        saved[f'{family}_{end}'] = scores
        if family == 'pairwise_ranker':
            bounded = expit(scores.astype(np.float64))
            assert abs(roc_auc_score(checking.label, bounded) - auc) < 1e-12
            saved[f'bounded_ranker_{end}'] = bounded
        row = dict(model=family, train_end=end, validation_start=first,
                   validation_end=last, learning_rows=len(learning),
                   checking_rows=len(checking), rounds=rounds, auc=auc,
                   seconds=round(time.perf_counter() - started, 3))
        records.append(row)
        print(f'{family}: learn 1–{end}; check {first}–{last}; '
              f'AUC={auc:.6f}; rounds={rounds}', flush=True)
    pd.DataFrame(records).to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
    np.savez_compressed(OUTPUT_DIR / 'validation_scores.npz', **saved)

results = pd.DataFrame(records)
comparison = results.pivot(index='train_end', columns='model', values='auc')
comparison['ranker_minus_original'] = comparison.pairwise_ranker - comparison.original_xgboost
summary = comparison[['original_xgboost', 'pairwise_ranker']].agg(['mean', 'min']).T
print('\nFour chronological checks:')
print(comparison.round(6).to_string())
print('\nMean and worst full-block AUC:')
print(summary.round(6).to_string())
display(comparison.round(6))
display(summary.round(6))
comparison.to_csv(OUTPUT_DIR / 'comparison.csv')
summary.to_csv(OUTPUT_DIR / 'summary.csv')
print('Comparison training seconds:', round(time.perf_counter() - started_all, 2))


original_xgboost: learn 1–14; check 15–28; AUC=0.968521; rounds=185
pairwise_ranker: learn 1–14; check 15–28; AUC=0.957249; rounds=198
original_xgboost: learn 1–17; check 18–31; AUC=0.925611; rounds=336
pairwise_ranker: learn 1–17; check 18–31; AUC=0.876898; rounds=142
original_xgboost: learn 1–19; check 20–33; AUC=0.901582; rounds=398
pairwise_ranker: learn 1–19; check 20–33; AUC=0.880458; rounds=258
original_xgboost: learn 1–21; check 22–35; AUC=0.915330; rounds=381
pairwise_ranker: learn 1–21; check 22–35; AUC=0.906442; rounds=173

Four chronological checks:
model      original_xgboost  pairwise_ranker  ranker_minus_original
train_end                                                          
14                 0.968521         0.957249              -0.011272
17                 0.925611         0.876898              -0.048714
19                 0.901582         0.880458              -0.021124
21                 0.915330         0.906442              -0.008888

Mean and worst full-blo

model,original_xgboost,pairwise_ranker,ranker_minus_original
train_end,,,
14,0.968521,0.957249,-0.011272
17,0.925611,0.876898,-0.048714
19,0.901582,0.880458,-0.021124
21,0.915330,0.906442,-0.008888


,mean,min
model,,
original_xgboost,0.927761,0.901582
pairwise_ranker,0.905262,0.876898


## 5. Where does it help or hurt?

Inspect scores within each time step as well as the pooled fourteen-step AUC. The pooled score remains the main comparison because the competition ranks all test transactions together. Overlapping checks do not provide independent estimates and must not be pooled by duplicating rows.

In [4]:
per_step_rows = []
for end, first, last in CHECKS:
    labels, steps = saved[f'labels_{end}'], saved[f'time_steps_{end}']
    for step in np.unique(steps):
        selected = steps == step
        row = {'train_end': end, 'validation_step': int(step),
               'known_labels': int(selected.sum()), 'illicit': int(labels[selected].sum())}
        for family in ['original_xgboost', 'pairwise_ranker']:
            row[family] = float(roc_auc_score(labels[selected], saved[f'{family}_{end}'][selected])) if len(np.unique(labels[selected])) == 2 else np.nan
        row['ranker_minus_original'] = row['pairwise_ranker'] - row['original_xgboost']
        per_step_rows.append(row)
per_step = pd.DataFrame(per_step_rows)
per_step.to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
print(per_step.loc[per_step.train_end == 21].round(6).to_string(index=False))
display(per_step.loc[per_step.train_end == 21].round(6))


 train_end  validation_step  known_labels  illicit  original_xgboost  pairwise_ranker  ranker_minus_original
        21               22          1763      158          0.998833         0.998632              -0.000201
        21               23          1187       53          0.957622         0.954378              -0.003244
        21               24          1126      137          0.795488         0.777516              -0.017971
        21               25           594      118          0.999484         0.999555               0.000071
        21               26           517       96          0.946779         0.935075              -0.011703
        21               27           206       24          0.999542         0.999313              -0.000229
        21               28           284       85          1.000000         1.000000               0.000000
        21               29          1174      329          0.775619         0.754346              -0.021273
        21         

,train_end,validation_step,known_labels,illicit,original_xgboost,pairwise_ranker,ranker_minus_original
42,21,22,1763,158,0.998833,0.998632,-0.000201
43,21,23,1187,53,0.957622,0.954378,-0.003244
44,21,24,1126,137,0.795488,0.777516,-0.017971
45,21,25,594,118,0.999484,0.999555,0.000071
46,21,26,517,96,0.946779,0.935075,-0.011703
47,21,27,206,24,0.999542,0.999313,-0.000229
48,21,28,284,85,1.000000,1.000000,0.000000
49,21,29,1174,329,0.775619,0.754346,-0.021273
50,21,30,524,83,0.987323,0.988225,0.000902
51,21,31,710,106,0.772820,0.787033,0.014213


## 6. Train the experimental candidate on all known labels

Preparing a separate CSV does not select it as the final model or use a Kaggle attempt. We train the fixed ranker on all known labels, with steps 33–35 choosing the round count before refitting on 1–35. Test labels are not available or used.

For the CSV, apply a sigmoid to the ranking scores. This preserves the order, gives values between 0 and 1, and does not claim calibrated probabilities. Match those values to the template by submission identifier. Save and reload the model to verify reproducibility.

In [5]:
final_ranker, final_rounds = fit_model('pairwise_ranker', known)
test = pd.read_csv(DATA_DIR / 'Test Data.csv')
template = pd.read_csv(DATA_DIR / 'Sample Submission' / 'BT4012 Competition Submission.csv')
assert test.txId.is_unique and test['index'].is_unique and template['index'].is_unique
assert set(test['index']) == set(template['index'])
assert len(test) == len(template)
assert known.time_step.max() < test.time_step.min()
assert set(known.txId).isdisjoint(test.txId)
assert np.isfinite(test[FEATURES].to_numpy()).all()
raw_test_scores = predict_scores('pairwise_ranker', final_ranker, test)
assert np.isfinite(raw_test_scores).all()
bounded_test_scores = expit(raw_test_scores.astype(np.float64))
assert np.all((bounded_test_scores > 0) & (bounded_test_scores < 1))

by_identifier = pd.Series(bounded_test_scores, index=test['index'])
submission = template[['index']].copy()
submission['target'] = submission['index'].map(by_identifier)
assert submission.columns.tolist() == ['index', 'target']
assert submission['index'].equals(template['index'])
assert np.isfinite(submission.target.to_numpy()).all()
assert submission.target.between(0, 1).all()
submission_path = OUTPUT_DIR / 'submission_xgboost_pairwise_experimental.csv'
model_path = OUTPUT_DIR / 'xgboost_pairwise_experimental.json'
submission.to_csv(submission_path, index=False, float_format='%.17g')
final_ranker.save_model(model_path)

restored = XGBRanker(n_jobs=CPU_THREADS)
restored.load_model(model_path)
restored_raw = predict_scores('pairwise_ranker', restored, test)
assert np.array_equal(restored_raw, raw_test_scores)
csv = pd.read_csv(submission_path)
assert csv['index'].equals(template['index'])
assert np.allclose(csv.target, submission.target, rtol=0, atol=1e-12)
np.savez_compressed(OUTPUT_DIR / 'test_scores.npz',
                    txIds=test.txId.to_numpy(), indices=test['index'].to_numpy(),
                    raw_scores=raw_test_scores, bounded_scores=bounded_test_scores)
print('Experimental submission rows:', len(submission))
print('Final training rounds:', final_rounds)
print('Test raw score range:', float(raw_test_scores.min()), float(raw_test_scores.max()))
print('Submission:', submission_path)
print('Saved model reload, identifiers, and CSV checks passed.')
display(submission.head())


Experimental submission rows: 15329
Final training rounds: 399
Test raw score range: -6.624268054962158 5.927134990692139
Submission: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/pairwise_ranking_results/submission_xgboost_pairwise_experimental.csv
Saved model reload, identifiers, and CSV checks passed.


,index,target
0,0,0.015552
1,1,0.038543
2,2,0.036419
3,3,0.023210
4,4,0.014686


## 7. Record settings, checks, and AI assistance

The student selected the pairwise-ranking objective and the stated fixed candidate, retaining the four chronological checks. OpenAI Codex implemented and ran that comparison, explained the learning objective, and prepared an experimental submission. The student must review the results, decide whether to submit it, and disclose the assistance accurately in the report.

The experimental file's Kaggle score is unknown. A new model must demonstrate improvement; novelty alone is not a reason to spend a submission attempt.

In [6]:
for end, first, last in CHECKS:
    expected = known.loc[known.time_step.between(first, last)]
    assert np.array_equal(saved[f'txIds_{end}'], expected.txId.to_numpy())
    for family in ['original_xgboost', 'pairwise_ranker']:
        auc = roc_auc_score(saved[f'labels_{end}'], saved[f'{family}_{end}'])
        assert abs(auc - comparison.loc[end, family]) < 1e-12

def file_hash(path):
    hasher = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

parameters = final_ranker.get_params()
parameters = {k: (None if isinstance(v, float) and not np.isfinite(v) else v)
              for k, v in parameters.items()
              if v is None or isinstance(v, (str, int, float, bool))}
metadata = dict(
    date='2026-10-11', student_decision='train and test the fixed pairwise-ranking candidate',
    checks=CHECKS, model_parameters=parameters, training_rounds=final_rounds,
    group_strategy='one ranking group per fitting partition; no outer-validation labels in fitting',
    stopping_metric='scikit-learn binary ROC-AUC, explicitly maximized and independently verified',
    initial_ranking_auc_mismatch_corrected=True,
    bounded_score_transform='sigmoid of raw ranking score; not calibrated probabilities',
    retained_features=FEATURES, known_labels=len(known), submission_rows=len(submission),
    ranking_mean_development_auc=float(summary.loc['pairwise_ranker', 'mean']),
    original_mean_development_auc=float(summary.loc['original_xgboost', 'mean']),
    outer_checks_overlap=True, outer_checks_are_development_evidence=True,
    original_public_auc=0.94265, candidate_public_auc=None,
    student_final_selection='pending review of results', kaggle_submission_made=False,
    python=platform.python_version(), xgboost=xgboost.__version__,
    pandas=pd.__version__, numpy=np.__version__,
    input_sha256={str(p.relative_to(DATA_DIR)): file_hash(p) for p in [
        DATA_DIR / 'Train Data.csv', DATA_DIR / 'Test Data.csv',
        DATA_DIR / 'Sample Submission' / 'BT4012 Competition Submission.csv']},
    submission_sha256=file_hash(submission_path), model_sha256=file_hash(model_path),
)
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2, allow_nan=False))
print('Chronological validation, alignment, transformation, saved-model and CSV checks passed.')
print('Final selection remains the student\'s decision. No Kaggle upload was made.')


Chronological validation, alignment, transformation, saved-model and CSV checks passed.
Final selection remains the student's decision. No Kaggle upload was made.


## 8. Measured result and submission review

This model was trained from scratch with the pairwise objective selected by the student. The corrected stopping calculation uses binary ROC-AUC on the inner period, explicitly maximizes it, and agrees with an independent calculation to the six-decimal reporting precision of XGBoost's Python custom metric. Outer comparison scores are calculated at full numerical precision.

| Validation steps | Original classifier | New ranking model | Change |
|---|---:|---:|---:|
| 15–28 | 0.968521 | 0.957249 | -0.011272 |
| 18–31 | 0.925611 | 0.876898 | -0.048714 |
| 20–33 | 0.901582 | 0.880458 | -0.021124 |
| 22–35 | 0.915330 | 0.906442 | -0.008888 |
| **Mean of four checks** | **0.927761** | **0.905262** | **-0.022499** |

The new model improved **0 of four** development checks. Its mean change was **-0.022499 AUC**. Its worst full-block AUC was 0.876898, versus 0.901582 for the classifier. These results compare the complete fixed candidate, including ranking default regularization; they do not isolate the training objective as the sole cause of any difference.

**AI-assisted recommendation:** The candidate did not improve the mean of the four chronological checks. I do not recommend spending a remaining submission on it based on these results. It should remain a documented experiment rather than replace the selected classifier.

The full-data candidate used **399 trees** and all **31,235 known labels** from steps 1–35. The separate experimental CSV contains **15,329 checked scores** for steps 36–49. The saved model reproduced the same predictions after reload. The sigmoid conversion preserved validation AUC, and submission identifiers and numeric values passed the checks.

The CSV is `pairwise_ranking_results/submission_xgboost_pairwise_experimental.csv`. It is an experimental candidate, not automatically selected as the final submission. No Kaggle upload was made.

The original XGBoost's confirmed public score remains **0.94265**. The new model's public and private scores are unknown. The local checks overlap, have influenced earlier development, and cannot demonstrate 0.96+ on hidden answers.

### What the student should explain

1. The earlier classifier trains to predict probabilities; AUC was its monitoring metric.
2. The new model trains on sampled illicit–licit orderings using pairwise logistic loss. This is a surrogate for ranking quality, not exact AUC optimization.
3. Training pairs use only known labels inside the fitting partition. Outer checking labels never enter training pairs or stopping selection.
4. A ranking score is not a calibrated fraud probability. The sigmoid conversion changes its range while preserving its order.
5. Comparing four chronological cutoffs helps detect dependence on a particular period. These overlapping cutoffs are still development evidence.
6. The initial built-in ranking AUC did not match binary ROC-AUC in this setup; correcting that implementation issue was necessary before judging the chosen model.
7. The final submission decision should account for improvements, declines, validation reuse, and the two remaining attempts reported by the student.

### AI usage and decision ownership

The student explicitly selected the pairwise-ranking experiment after its objective, settings, and limitations were explained, retaining the previously selected validation strategy. OpenAI Codex implemented the experiment, diagnosed and corrected the stopping metric mismatch, ran the fixed comparison, verified the artifacts, and prepared this journal entry. The student must review the interpretation, choose whether to submit this candidate, and accurately disclose the assistance in the report. No autonomous parameter search or final-model selection was performed.
